# Clase 24: CNN — De MLP a Convoluciones

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador x UDLA

---

**Objetivos de hoy:**
1. Entender por qué el MLP **falla** en imágenes (cats vs dogs de la clase 22).
2. Conocer la **operación convolución** y por qué resuelve el problema.
3. Aprender por qué usamos **Keras** (vs TensorFlow puro o PyTorch).
4. Construir nuestra **primera CNN** en MNIST → ~99% en 5 epochs.
5. Aplicarla a **cats vs dogs** y romper el techo del 60% al 80%+.
6. Conocer **transfer learning** y cuándo NO usar CNN.

## 0. Imports

> **Nota:** este notebook usa TensorFlow/Keras. En Colab ya viene preinstalado. Local: `pip install tensorflow`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, warnings, urllib.request, io
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU'))} dispositivo(s)")
print("(En Colab gratis hay GPU si activas Runtime -> Change runtime type -> T4 GPU)")

---
## 0.5 Pre-vuelo: cómo Keras ve las imágenes

Antes de entrenar nada, hay que entender **cómo Keras espera los datos**. Casi todos los errores de principiante con CNN vienen de no tener claro esto.

Una imagen tiene **siempre 3 dimensiones**:

| Dimensión | Significado | Ejemplo MNIST | Ejemplo RGB |
|-----------|-------------|---------------|-------------|
| H (altura) | filas de píxeles | 28 | 96 |
| W (ancho) | columnas de píxeles | 28 | 96 |
| C (canales) | 1 = grises, 3 = RGB | 1 | 3 |

Al **entrenar** procesamos muchas imágenes a la vez (un *batch*), así que el array tiene **4 dimensiones**: `(N, H, W, C)`.

**Las dos cosas más importantes:**
1. Si tu imagen es en grises (`shape (28, 28)`), tienes que **agregar el canal**: `img[..., None]` → `(28, 28, 1)`. Si no, Conv2D te da error.
2. El parámetro `input_shape` se especifica **SOLO en la primera capa**. Las siguientes lo infieren automáticamente.

In [ ]:
# Ejemplo: cómo agregar la dimensión de canal
import numpy as np

img = np.zeros((28, 28))
print(f"Imagen sola:           {img.shape}  <- una CNN da error con esto")

# 3 formas equivalentes de agregar el canal
img1 = img[..., None]
img2 = img[:, :, np.newaxis]
img3 = np.expand_dims(img, axis=-1)
print(f"Con canal agregado:    {img1.shape}  <- ahora la CNN la acepta")

# Para entrenar: agregar tambien la dimension de batch (al inicio)
batch = np.stack([img, img, img])         # 3 imagenes
batch_with_ch = batch[..., None]
print(f"Batch de 3 imagenes:   {batch_with_ch.shape}  (N, H, W, C)")

**Resumen del orden de dimensiones:**

| Lo que tienes | Shape | Lo que necesitas |
|---------------|-------|-------------------|
| 1 imagen grises | `(28, 28)` | `(28, 28, 1)` |
| 1 imagen RGB | `(96, 96, 3)` | OK, ya tiene canal |
| Lote para entrenar | `(N, 28, 28)` | `(N, 28, 28, 1)` |
| Predecir UNA imagen | `(28, 28, 1)` | `(1, 28, 28, 1)` con `img[None, ...]` |

> **Regla de oro:** Keras siempre espera **4 dimensiones** al entrenar/predecir, incluso si solo tienes 1 imagen.

---
## 1. TensorFlow Playground (5 min — abre el navegador)

Antes de escribir código, **construyamos una red en vivo**.

🌐 **Abrir:** https://playground.tensorflow.org

### 4 mini-experimentos

| # | Dataset | Configuración | Lo que aprendemos |
|---|---------|----------------|-------------------|
| 1 | Círculos | 1 capa de 4, Tanh, solo $x_1, x_2$ | Una sola capa con activación no lineal **ya curva** |
| 2 | Espiral | 1 capa de 8 → falla; 3 capas de 8 → funciona | **Profundidad** importa cuando la frontera es compleja |
| 3 | Espiral | Cambiar entre Sigmoid / Tanh / ReLU | ReLU es la más limpia y rápida |
| 4 | Círculos | Activar solo $x_1^2$ y $x_2^2$ | Si las **features** son las correctas, una sola neurona basta |

**Lección clave:** las redes **aprenden features automáticamente**. Cada capa aprende algo más abstracto que la anterior. Hoy lo veremos concretamente con imágenes.

---
## 2. Recordemos el problema: cats vs dogs

En la clase 22 entrenamos Logistic, RF y MLP sobre el mismo dataset CIFAR-10 cats vs dogs (32×32 RGB). Los resultados fueron:

| Modelo | MNIST (28×28 grises) | Cats vs Dogs (32×32 RGB) |
|--------|----------------------|---------------------------|
| Logistic | 92% | **~55%** (azar) |
| Random Forest | 95% | **~62%** |
| MLP (128, 64) | 97% | **~60%** |

Aumentar capacidad **no rompe la pared**. Veamos por qué.

### 2.1 Por qué falla: el MLP no es invariante a traslación

Si movemos un dígito **1 pixel a la derecha**, el vector aplanado cambia *completamente*. El MLP tiene que aprender "ese 3" para cada posición posible.

In [ ]:
from sklearn.datasets import load_digits
from scipy.ndimage import shift

digits = load_digits()
img3 = digits.images[np.where(digits.target == 3)[0][0]]   # 8x8 '3'

shifts = [0, 1, 2, 3, 4]
fig, axes = plt.subplots(2, 5, figsize=(13, 5))

for i, s in enumerate(shifts):
    shifted = shift(img3, [0, s], mode="constant", cval=0)
    axes[0, i].imshow(shifted, cmap="gray_r")
    axes[0, i].set_title(f"'3' desplazado +{s}px", fontweight="bold")
    axes[0, i].axis("off")

    flat = shifted.flatten()
    axes[1, i].imshow(flat.reshape(1, -1), cmap="gray_r", aspect="auto")
    diff = np.linalg.norm(flat - img3.flatten())
    color = "green" if diff == 0 else ("orange" if diff < 30 else "red")
    axes[1, i].set_title(f"vector | dist={diff:.0f}", color=color, fontweight="bold")
    axes[1, i].set_xticks([]); axes[1, i].set_yticks([])

plt.suptitle("MLP ve un vector totalmente distinto cuando movemos el digito 1 pixel",
             fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

**Interpretación:** la distancia euclidiana entre el vector original y el desplazado crece rápido. Para nosotros es el mismo "3"; para el MLP son ejemplos completamente distintos.

### 2.2 Por qué falla: explosión de parámetros

Cada pixel se conecta con **cada** neurona oculta. Con imágenes grandes esto explota:

In [ ]:
hidden = 128
sizes = [(28, 28, 1, "MNIST 28x28"),
         (32, 32, 3, "CIFAR 32x32 RGB"),
         (96, 96, 3, "Foto pequena 96x96"),
         (224, 224, 3, "Foto web 224x224")]

print(f"Parametros en la 1ra capa con {hidden} neuronas ocultas:\n")
for h, w, c, name in sizes:
    n = h * w * c * hidden + hidden
    print(f"  {name:25s} -> {n:>15,} parametros")

**Interpretación:** una sola capa oculta de 128 neuronas para una foto 224×224×3 requiere casi **20 millones** de parámetros. Aprenderlos pide muchos más datos de los que normalmente tenemos.

---
## 3. La solución: filtros que se *comparten*

En vez de un peso por pixel, usamos los **mismos** 9 pesos (filtro 3×3) **moviéndose** por la imagen. Este es el corazón de la CNN:

- **Pocos parámetros** (9 vs miles).
- **Invariante a posición** automáticamente: si el filtro detecta un borde en la esquina, también lo detecta en el centro.
- **Captura estructura local**: solo mira pixeles vecinos a la vez.

Antes de las CNN, los ingenieros diseñaban filtros **a mano**. La CNN simplemente los **aprende** de los datos.

In [ ]:
from sklearn.datasets import load_sample_images
from scipy.ndimage import convolve

samples = load_sample_images()
img_color = samples.images[1]    # flor RGB
img_gray = img_color.mean(axis=2)
H, W = 280, 280
cy, cx = img_gray.shape[0]//2, img_gray.shape[1]//2
img_gray = img_gray[cy-H//2:cy+H//2, cx-W//2:cx+W//2]

# Filtros a mano (los mismos que aprende una CNN, pero hardcoded)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
sobel_y = sobel_x.T
edges = np.sqrt(convolve(img_gray, sobel_x)**2 + convolve(img_gray, sobel_y)**2)
blur = convolve(img_gray, np.ones((5, 5))/25)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_gray, cmap="gray"); axes[0].set_title("Input (grises)"); axes[0].axis("off")
axes[1].imshow(edges, cmap="gray"); axes[1].set_title("Filtro Sobel = bordes", color="red"); axes[1].axis("off")
axes[2].imshow(blur, cmap="gray"); axes[2].set_title("Filtro promedio = desenfoque", color="blue"); axes[2].axis("off")
plt.tight_layout(); plt.show()

print("Sobel  =", sobel_x.flatten().tolist(), "  (9 numeros, detecta bordes verticales)")
print("Blur   = matriz 5x5 con valores 1/25  (25 numeros, promedia)")

### 3.1 La operación convolución, paso a paso

1. Tomar una ventana 3×3 de la imagen.
2. Multiplicar elemento a elemento con el filtro.
3. Sumar todos los productos → un solo número.
4. Mover la ventana 1 posición y repetir.

In [ ]:
# Demo numerica de la convolucion
input_img = np.array([
    [3, 1, 2, 7, 5],
    [0, 1, 1, 4, 8],
    [1, 2, 2, 0, 1],
    [3, 9, 8, 0, 0],
    [4, 8, 1, 5, 4]
])
kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]])    # detector de borde vertical

# Aplicar manualmente (sin scipy) para entender la mecanica
out = np.zeros((3, 3), dtype=int)
for i in range(3):
    for j in range(3):
        ventana = input_img[i:i+3, j:j+3]
        out[i, j] = (ventana * kernel).sum()

print("INPUT (5x5):");  print(input_img)
print("\nFILTRO (3x3):"); print(kernel)
print("\nOUTPUT (3x3):"); print(out)
print("\n-> Cada numero del output = 9 multiplicaciones + 1 suma de la ventana")

---
## 4. Keras: ¿qué es y por qué la usamos?

**Keras** es una **API de alto nivel** para construir redes neuronales. No hace los cálculos: pide ayuda a un *backend* (TensorFlow, JAX, PyTorch).

```
Tu código (Python)
       ↓
     Keras           ← API limpia: Sequential, Conv2D, fit, predict
       ↓
TensorFlow / JAX     ← backend: gradientes, GPU
       ↓
   GPU / CPU
```

### ¿Por qué Keras y no PyTorch puro?

| | **Keras** | **PyTorch puro** |
|---|---|---|
| API | `model.fit(X, y)` (igual que sklearn) | Loop manual: `for epoch`, `loss.backward()`, `optimizer.step()` |
| Para empezar | ✅ Mínima fricción | Más líneas para lo básico |
| Investigación / custom | OK con la API funcional | ✅ Más control |
| Compatibilidad sklearn | ✅ Pipeline, GridSearch | Hay que adaptar |

**Conclusión pedagógica:** aprenden Keras hoy → transfieren a PyTorch sin problema (los conceptos son los mismos). PyTorch es el siguiente paso natural cuando quieran construir modelos custom complejos.

---
## 5. Primera CNN: MNIST en 5 epochs

Ahora la receta. Misma API que sklearn, solo que primero hay que `compile()` antes de `fit()`.

In [ ]:
# Cargar MNIST con Keras (cache automatico)
(X_tr_m, y_tr_m), (X_te_m, y_te_m) = keras.datasets.mnist.load_data()
print(f"Train: {X_tr_m.shape}, Test: {X_te_m.shape}")

# Agregar canal: (28, 28) -> (28, 28, 1) y normalizar
X_tr_m = X_tr_m[..., None].astype("float32") / 255.0
X_te_m = X_te_m[..., None].astype("float32") / 255.0
print(f"Despues: train {X_tr_m.shape}, valores en [0, 1]")

In [ ]:
# Construir la CNN
cnn_mnist = Sequential([
    layers.Conv2D(32, 3, activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="CNN_MNIST")

cnn_mnist.summary()

### Cómo leer `model.summary()` y por qué `input_shape=(28, 28, 1)`

Mira la columna `Output Shape`: para la primera capa aparece algo como `(None, 26, 26, 32)`.

| Parte | Significado |
|-------|-------------|
| **`None`** | El batch (cantidad de imágenes). Keras no sabe cuántas vas a meterle, por eso `None`. |
| **`26, 26`** | Alto y ancho del feature map después de Conv2D. |
| **`32`** | Cantidad de canales (1 por filtro de Conv2D 32). |

**¿Por qué pasamos de 28 a 26?** Sin padding, una ventana 3×3 no llega hasta los bordes (la ventana se sale). Pierde 2 pixeles (uno por cada lado). Con `padding="same"` se conservaría 28×28.

**¿Por qué pasamos de 26 a 13?** MaxPooling 2×2 reduce a la mitad cada dimensión.

**Patrón a observar:**
- Cada **Conv2D** mantiene (o reduce un poco) la resolución y aumenta los canales.
- Cada **MaxPooling** reduce la resolución a la mitad sin tocar los canales.
- Al final el `Flatten` desenrolla todo en un vector grande.
- Las `Dense` finales reducen a la cantidad de clases.

**Truco mental:** mientras más profundo en la red, más canales (32 → 64 → 128) y menos resolución (28 → 13 → 5).

### Interpretación del conteo de parámetros

- `Conv2D 32 filtros 3×3` tiene solo **320 parámetros** ($32 \times (3 \times 3 \times 1 + 1)$). Eso es parameter sharing.
- `MaxPooling` no tiene parámetros (es una operación fija).
- `Dense 64` (después del Flatten) es la que más parámetros tiene — la parte "MLP" del modelo.
- **Total: ~120K parámetros**, comparable con un MLP pequeño y mejor accuracy.

### `compile()`: el "modo de entrenamiento" del modelo

Antes de entrenar, hay que decirle al modelo **3 cosas**:

| Argumento | Qué decide |
|-----------|------------|
| `optimizer` | Cómo se ajustan los pesos en cada paso. **Adam** es el default moderno, casi siempre funciona. |
| `loss` | Qué función usar para medir el "error" del modelo. Depende del problema (ver tabla abajo). |
| `metrics` | Qué reportar durante el entrenamiento (legible para humanos). |

**Las 3 funciones de loss más comunes (clasificación):**

| Loss | Cuándo usar | Cómo lucen las etiquetas |
|------|-------------|---------------------------|
| `binary_crossentropy` | **2 clases**, capa final con 1 neurona + sigmoid | `0` o `1` |
| `categorical_crossentropy` | **N clases**, etiquetas en formato **one-hot** | `[0, 0, 1, 0, 0]` |
| `sparse_categorical_crossentropy` | **N clases**, etiquetas como **enteros** | `2` (significa clase 2) |

> **¿Por qué usamos `sparse_`?** Porque nuestras etiquetas son enteros (`y_tr_m` tiene valores 0..9), no one-hot. La palabra **sparse** ("disperso") es porque internamente el cálculo evita expandir el one-hot — es una optimización.

> **¿Cuándo usar el otro?** Si tus etiquetas ya están en one-hot (ej. `y = [[0,1,0], [1,0,0], ...]`), usa `categorical_crossentropy`. Mismo resultado, distinto formato de entrada.

In [ ]:
# Compile = decirle al modelo: optimizer, loss, metricas
cnn_mnist.compile(
    optimizer="adam",                            # ajustador de pesos por defecto
    loss="sparse_categorical_crossentropy",      # multiclase con etiquetas enteras
    metrics=["accuracy"],
)
print("Modelo compilado. Listo para .fit()")

In [ ]:
# Entrenar - misma API que sklearn, solo cambia que primero hay que compilar
t0 = time.time()
history_mnist = cnn_mnist.fit(
    X_tr_m, y_tr_m,
    epochs=5,
    batch_size=128,            # cuantas imagenes procesa en cada paso
    validation_split=0.1,      # 10% para validacion (separado del train)
    verbose=2,
)
t_train = time.time() - t0

acc_test = cnn_mnist.evaluate(X_te_m, y_te_m, verbose=0)[1]
print(f"\nAccuracy en TEST: {acc_test:.4f}")
print(f"Tiempo de entrenamiento: {t_train:.1f}s")

In [ ]:
# Curva de loss y accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_mnist.history["loss"], label="train", color="#C82B40")
axes[0].plot(history_mnist.history["val_loss"], label="val", color="#2563EB")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss")
axes[1].plot(history_mnist.history["accuracy"], label="train", color="#C82B40")
axes[1].plot(history_mnist.history["val_accuracy"], label="val", color="#2563EB")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
axes[1].set_title("Accuracy")
plt.tight_layout(); plt.show()

---
## 5.5 Resize: preparar imágenes a un tamaño común

Cuando colectas fotos del mundo real (con celular, webcam, internet), cada una puede tener un tamaño distinto: 4032×3024, 1080×1920, 800×600... Pero una CNN **siempre espera el mismo shape** en sus inputs.

**Hay que hacer resize a TODAS las imágenes a un tamaño único antes de entrenar.**

### ¿Cuál tamaño elegir?

| Tamaño | Cuándo usarlo |
|--------|----------------|
| 32×32 | Datasets clásicos (CIFAR, MNIST). Muy rápido pero pierde detalle. |
| 64×64 | Prototipado rápido con poca data. |
| **96×96** | **Bueno con transfer learning.** Mínimo de MobileNet/EfficientNet. |
| 224×224 | Estándar de ImageNet. Lo que VGG16 y ResNet esperan idealmente. |

**Compromiso:** más grande = más detalle pero más cómputo. Para nuestros proyectos **96×96** es un buen equilibrio.

In [ ]:
# Demo: tomar una imagen real y hacer resize a distintos tamanos
# Usamos una de las imagenes de muestra de sklearn (no requiere internet)
from sklearn.datasets import load_sample_images

img_orig = load_sample_images().images[1]    # foto de una flor (427, 640, 3)
print(f"Imagen original: {img_orig.shape}")

# Resize con tf.image.resize a distintos tamanos
img_224 = tf.image.resize(img_orig, (224, 224)).numpy().astype(np.uint8)
img_96  = tf.image.resize(img_orig, (96, 96)).numpy().astype(np.uint8)
img_32  = tf.image.resize(img_orig, (32, 32)).numpy().astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, t in zip(axes,
    [img_orig, img_224, img_96, img_32],
    [f"Original {img_orig.shape[:2]}", "224x224", "96x96", "32x32"]):
    ax.imshow(im); ax.set_title(t, fontweight="bold"); ax.axis("off")
plt.tight_layout(); plt.show()

**Observación importante:** mientras más pequeño el resize, más detalle se pierde. A 32×32 una lata es casi un cuadrado. A 224×224 todavía se ve la marca.

### Métodos de resize

| Método | Qué hace | Cuándo usarlo |
|--------|----------|---------------|
| **Stretch** (default `tf.image.resize`) | Estira la imagen al tamaño objetivo, puede distorsionar | Si las proporciones no son críticas |
| **Crop center** | Recorta un cuadrado central | Si el objeto está en el centro |
| **Pad** | Agrega bordes negros para mantener proporciones | Si nada se debe perder |

```python
# Stretch (lo más simple, lo que usamos por defecto)
img = tf.image.resize(img, (96, 96))

# Crop center cuadrado
img = tf.image.resize_with_crop_or_pad(img, 96, 96)

# Mantener proporciones, agregar pad
img = tf.image.resize_with_pad(img, 96, 96)
```

**Para nuestros datasets pequeños el stretch funciona bien.** Si tus objetos son alargados (botellas) o tienen aspect ratio crítico, considera crop o pad.

---
## 6. CNN en cats vs dogs — el payoff

Misma data de la clase 22 (4,000 imágenes 32×32 RGB de gatos y perros). En la clase 22 nuestros modelos llegaron a ~60%. Veamos qué hace una CNN.

In [ ]:
# Cargar el subset desde el repo de la clase 22
URL_NPZ = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-22/cifar_cats_dogs_4k.npz"
with urllib.request.urlopen(URL_NPZ, timeout=30) as r:
    data = np.load(io.BytesIO(r.read()))
X_cd, y_cd = data["X"], data["y"]
print(f"X_cd: {X_cd.shape}, y_cd: {y_cd.shape}, balance: {np.bincount(y_cd)}")

# Normalizar y splitear
X_cd_n = X_cd.astype("float32") / 255.0
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cd_n, y_cd, test_size=0.2, random_state=42, stratify=y_cd)
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

In [ ]:
cnn_cd = Sequential([
    layers.Conv2D(32, 3, activation="relu", padding="same", input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),                  # regularizacion: apaga aleatoriamente neuronas en train
    layers.Dense(64, activation="relu"),
    layers.Dense(2, activation="softmax"),
], name="CNN_CatsDogs")

cnn_cd.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
cnn_cd.summary()

In [ ]:
t0 = time.time()
history_cd = cnn_cd.fit(
    X_tr, y_tr,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=2,
)
print(f"\nTiempo: {time.time()-t0:.1f}s")
acc_cd = cnn_cd.evaluate(X_te, y_te, verbose=0)[1]
print(f"Accuracy en TEST: {acc_cd:.3f}  (objetivo: superar el 60% del MLP de la clase 22)")

In [ ]:
# Comparar con clase 22
clase22 = {"Logistic": 0.55, "Random Forest": 0.62, "MLP (128, 64)": 0.60}
clase24 = {**clase22, "CNN (esta clase)": acc_cd}

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["#9CA3AF", "#9CA3AF", "#9CA3AF", "#C82B40"]
bars = ax.bar(clase24.keys(), clase24.values(), color=colors)
for b, v in zip(bars, clase24.values()):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.0%}",
            ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.axhline(0.5, ls="--", color="gray", lw=1, label="azar")
ax.set_ylabel("Accuracy"); ax.set_ylim(0.4, 1.0)
ax.set_title("Cats vs Dogs: la CNN cierra el gap", fontweight="bold")
ax.legend()
plt.tight_layout(); plt.show()

**Interpretación:** con los **mismos datos** que en la clase 22, la CNN salta de ~60% a ~80%+. La diferencia es **toda** arquitectural: filtros + parameter sharing + estructura espacial.

### 6.1 ¿Qué aprendieron los filtros?

Podemos extraer los pesos de la primera capa convolucional para ver los filtros que la CNN inventó.

In [ ]:
# Extraer los filtros de la primera Conv2D
W_filtros = cnn_cd.layers[0].get_weights()[0]   # shape: (3, 3, 3, 32)
print(f"Forma de los pesos: {W_filtros.shape}")
print(f"  3x3 = tamano del filtro,  3 = canales RGB de entrada,  32 = filtros aprendidos")

# Mostrar 16 filtros (cada uno es un cubito 3x3x3 que vemos como pseudo-RGB)
fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i, ax in enumerate(axes.flat):
    f = W_filtros[..., i]
    # Normalizar a [0, 1] solo para visualizar
    f_vis = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f_vis)
    ax.axis("off")
    ax.set_title(f"f{i}", fontsize=8)
plt.suptitle("Algunos de los 32 filtros 3x3 que la CNN APRENDIO\n(colores raros porque no son imagenes reales, son pesos)",
             fontweight="bold", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Aplicar los filtros aprendidos a una imagen real
demo_img = X_tr[0:1]   # (1, 32, 32, 3)

# Crear un sub-modelo que solo va hasta la primera Conv2D
extractor = Sequential([cnn_cd.layers[0]])    # solo la primera capa
feature_maps = extractor.predict(demo_img, verbose=0)[0]    # (32, 32, 32)
print(f"Feature maps shape: {feature_maps.shape}  -> 32 mapas de 32x32")

fig, axes = plt.subplots(3, 8, figsize=(13, 5))
axes[0, 0].imshow(demo_img[0]); axes[0, 0].set_title("input"); axes[0, 0].axis("off")
for j in range(1, 8): axes[0, j].axis("off")
for i in range(16):
    ax = axes[1 + i // 8, i % 8]
    ax.imshow(feature_maps[..., i], cmap="viridis")
    ax.axis("off")
    ax.set_title(f"map {i}", fontsize=8)
plt.suptitle("Mapas de caracteristicas que produce la 1ra Conv2D sobre una foto real",
             fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

**Interpretación:** cada mapa es la respuesta de un filtro distinto. Algunos resaltan **bordes verticales**, otros **horizontales**, otros **texturas** específicas. La CNN inventó esto solita por gradient descent.

---
## 7. Data augmentation — más datos sin recolectar más datos

Truco esencial cuando hay pocos datos: generar variaciones (rotación, flip, brillo) y usar todas en entrenamiento. Keras lo hace agregando capas de augmentation **al inicio del modelo** (solo se activan en training).

In [ ]:
# Capa de data augmentation (se aplica solo durante .fit, no en .predict)
data_aug = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),     # +/- 10%
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
], name="augmentation")

# Visualizar 5 versiones aumentadas de una misma imagen
fig, axes = plt.subplots(1, 6, figsize=(13, 3))
axes[0].imshow(X_tr[10]); axes[0].set_title("Original"); axes[0].axis("off")
for i in range(1, 6):
    aug = data_aug(X_tr[10:11], training=True)[0]
    axes[i].imshow(np.clip(aug.numpy(), 0, 1))
    axes[i].set_title(f"Augmentado {i}"); axes[i].axis("off")
plt.suptitle("La misma foto vista de muchas formas (mas datos gratis)",
             fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# CNN con augmentation incorporada — la receta moderna
cnn_aug = Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_aug,                                            # ahora forma parte del modelo
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(2, activation="softmax"),
])
cnn_aug.compile(optimizer="adam",
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])

t0 = time.time()
history_aug = cnn_aug.fit(X_tr, y_tr, epochs=20, batch_size=64,
                          validation_split=0.1, verbose=2)
acc_aug = cnn_aug.evaluate(X_te, y_te, verbose=0)[1]
print(f"\nAccuracy con augmentation: {acc_aug:.3f}  (vs {acc_cd:.3f} sin)")

---
## 8. Transfer learning — la solución cuando hay pocos datos

**Idea:** tomar una CNN ya entrenada en ImageNet (1.4 millones de imágenes), congelar sus capas convolucionales, y entrenar solo un cabezal nuevo con tus datos.

Los filtros de las primeras capas (bordes, texturas) son **universales**. Lo único "de tu problema" es la última decisión.

In [ ]:
# MobileNetV2: CNN pequena pero potente, preentrenada en ImageNet
# CIFAR es 32x32 -> upsampleamos a 96x96 para que MobileNet funcione bien
from tensorflow.keras.applications import MobileNetV2

# Resize 32x32 -> 96x96 (MobileNetV2 minimo)
X_tr_big = tf.image.resize(X_tr, (96, 96)).numpy()
X_te_big = tf.image.resize(X_te, (96, 96)).numpy()
print(f"Resized train: {X_tr_big.shape}")

base = MobileNetV2(input_shape=(96, 96, 3),
                   include_top=False,
                   weights="imagenet")
base.trainable = False

print(f"\nMobileNetV2 cargado. Parametros: {base.count_params():,}")

In [ ]:
# Construir el modelo de transfer learning
modelo_xfer = Sequential([
    base,
    layers.GlobalAveragePooling2D(),       # alternativa moderna a Flatten + Dense
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(2, activation="softmax"),
])

modelo_xfer.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

# Solo se entrena el cabezal: pocos parametros, pocas epochs
trainable = sum(np.prod(v.shape) for v in modelo_xfer.trainable_variables)
print(f"Parametros entrenables: {trainable:,}  (vs {modelo_xfer.count_params():,} totales)")

In [ ]:
t0 = time.time()
history_xfer = modelo_xfer.fit(X_tr_big, y_tr, epochs=5, batch_size=64,
                               validation_split=0.1, verbose=2)
acc_xfer = modelo_xfer.evaluate(X_te_big, y_te, verbose=0)[1]
print(f"\nAccuracy con transfer learning: {acc_xfer:.3f}")
print(f"Tiempo: {time.time()-t0:.1f}s")

In [ ]:
# Comparacion final
final_results = {
    "Logistic (clase 22)": 0.55,
    "Random Forest (clase 22)": 0.62,
    "MLP (clase 22)": 0.60,
    "CNN desde cero": acc_cd,
    "CNN + Augmentation": acc_aug,
    "Transfer Learning (MobileNetV2)": acc_xfer,
}

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ["#9CA3AF"] * 3 + ["#EA580C", "#C82B40", "#16A34A"]
labels = list(final_results.keys())
values = list(final_results.values())
bars = ax.barh(labels, values, color=colors)
for b, v in zip(bars, values):
    ax.text(v + 0.005, b.get_y() + b.get_height()/2, f"{v:.0%}",
            va="center", fontweight="bold", fontsize=10)
ax.axvline(0.5, ls="--", color="gray", lw=1, label="azar")
ax.set_xlim(0.4, 1.0); ax.set_xlabel("Accuracy"); ax.legend()
ax.set_title("Cats vs Dogs: la jerarquia de soluciones", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 9. Cuándo usar (y NO usar) una CNN

### Las CNN no son gratis: 4 costos reales

| Costo | Cómo se manifiesta |
|-------|--------------------|
| **Datos** | CNN seria pide decenas de miles de imágenes etiquetadas |
| **Cómputo** | GPU prácticamente obligatoria para datasets grandes |
| **Hiperparámetros** | Filtros, profundidad, learning rate, dropout, scheduler… mucho tuning |
| **Interpretabilidad** | Difícil decir "por qué" predijo perro |

### Decisión rápida

| Caso | Recomendación |
|------|----------------|
| Imágenes, > 10K etiquetadas | CNN desde cero (o transfer + fine-tune) |
| Imágenes, < 10K etiquetadas | **Transfer learning siempre** |
| Tabular | RF / XGBoost / LightGBM |
| Texto | Transformers / LLMs |
| Series temporales | Transformers / LSTM |
| Audio | CNN 1D o 2D sobre espectrograma |

### CNN vs Vision Transformers (ViT)

Desde 2020 las **Vision Transformers (ViT)** compiten y a veces superan a las CNN. Modelos modernos (CLIP, DINOv2, SAM, Stable Diffusion) son a base de transformers. **Pero las CNN siguen siendo el caballo de batalla** cuando hay poca data, pocos recursos o restricciones de latencia (móviles, IoT, líneas de producción).

**Recomendación práctica para Arca:** transfer learning sobre MobileNetV2/V3 o EfficientNet. Cubre el 90% de los casos de visión industrial: control de calidad, lectura de etiquetas, detección de productos.

---
## 10. Proyecto Gradio: Coca-Cola vs Pepsi

Vamos a construir el flujo completo de un proyecto de visión real:

1. **App de recolección** — Gradio captura fotos con la webcam y las guarda etiquetadas
2. **Cargar y preprocesar** — leer las fotos, resize a 96×96
3. **Entrenar** — transfer learning con VGG16
4. **App de predicción** — Gradio recibe foto y devuelve la clase con probabilidad

> ⚠️ Esta sección requiere `gradio`. En Colab: `!pip install gradio`.

In [ ]:
!pip install -q gradio
import gradio as gr
import os
from pathlib import Path

### 9.1 Crear la estructura de carpetas

Cada clase tendrá su propia carpeta. Esa convención (`dataset/clase_nombre/img_001.jpg`) es estándar y la entiende `keras.utils.image_dataset_from_directory` directamente.

In [ ]:
DATA_DIR = "dataset_latas"
class_names = ["coca_cola", "pepsi"]

for clase in class_names:
    os.makedirs(f"{DATA_DIR}/{clase}", exist_ok=True)
print(f"Carpetas listas:")
for clase in class_names:
    print(f"  {DATA_DIR}/{clase}/")

### 9.2 App de recolección con webcam

Esta app captura fotos con la webcam y las guarda en la carpeta de la clase elegida. Incluye un contador para que sepan cuántas fotos llevan.

In [ ]:
def guardar_foto(imagen, clase):
    if imagen is None:
        return "No hay foto. Toma una con la webcam."
    folder = f"{DATA_DIR}/{clase}"
    n = len(os.listdir(folder))
    fname = f"{folder}/img_{n:03d}.jpg"
    Image.fromarray(imagen).save(fname)
    counts = {c: len(os.listdir(f"{DATA_DIR}/{c}")) for c in class_names}
    return f"Guardada: {fname}\nTotal: " + " | ".join(f"{c}: {counts[c]}" for c in class_names)

with gr.Blocks(title="Recolector de latas") as recolector:
    gr.Markdown("# Recolector: Coca-Cola vs Pepsi")
    gr.Markdown("Selecciona la clase, toma una foto y haz click en Guardar. **Mínimo 30 fotos por clase**, variando ángulo, distancia y fondo.")
    with gr.Row():
        webcam = gr.Image(sources=["webcam"], type="numpy", label="Webcam")
        with gr.Column():
            clase = gr.Radio(class_names, label="Clase de la lata", value="coca_cola")
            btn = gr.Button("Guardar foto", variant="primary", size="lg")
            output = gr.Textbox(label="Estado", lines=3)
    btn.click(guardar_foto, inputs=[webcam, clase], outputs=output)

recolector.launch(share=True, debug=False)

**Mientras los estudiantes recolectan**, hay que pensar en la **diversidad** del dataset:
- Tomar fotos desde **distintos ángulos** (frontal, lateral, arriba)
- Usar **fondos distintos** (mesa, mano, escritorio)
- Variar **iluminación** (luz natural, fluorescente)
- **Distintas distancias** y orientaciones de la lata

**Sin esta diversidad, el modelo memoriza el fondo en vez de aprender la lata.**

Cuando todos hayan recolectado, ejecuten las celdas siguientes.

### 9.3 Cargar las fotos recolectadas y preprocesar

In [ ]:
# Leer todas las fotos, hacer resize a 96x96, normalizar
X_imgs, y_labels = [], []

for label, clase in enumerate(class_names):
    folder = Path(DATA_DIR) / clase
    files = sorted(folder.glob("*.jpg"))
    print(f"  {clase}: {len(files)} fotos")
    for fp in files:
        img = np.array(Image.open(fp).convert("RGB"))
        img_resized = tf.image.resize(img, (96, 96)).numpy()
        X_imgs.append(img_resized)
        y_labels.append(label)

X = np.array(X_imgs, dtype="float32") / 255.0     # normalizar [0, 1]
y = np.array(y_labels)
print(f"\nTotal: {len(X)} fotos")
print(f"X.shape = {X.shape}   <- (N, H, W, C)")
print(f"Balance: {dict(zip(class_names, np.bincount(y)))}")

In [ ]:
# Train / test split estratificado (mismo balance en ambos)
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")
print(f"Balance train: {dict(zip(class_names, np.bincount(y_tr)))}")
print(f"Balance test:  {dict(zip(class_names, np.bincount(y_te)))}")

In [ ]:
# Visualizar algunas fotos por clase para verificar que se cargaron bien
fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for k, clase in enumerate(class_names):
    idxs = np.where(y == k)[0][:6]
    for j, idx in enumerate(idxs):
        ax = axes[k, j]
        ax.imshow(X[idx])
        ax.axis("off")
        if j == 0: ax.set_ylabel(clase, fontsize=11, fontweight="bold")
plt.suptitle("Muestras del dataset recolectado", fontweight="bold")
plt.tight_layout(); plt.show()

### 9.4 Entrenar el modelo (transfer learning con VGG16)

Como tenemos pocos datos (~60 fotos), entrenar desde cero no funciona. Usamos VGG16 preentrenado en ImageNet como backbone congelado, agregamos data augmentation (rotación, flip, brillo) para multiplicar la diversidad, y entrenamos solo un cabezal pequeño.

In [ ]:
from tensorflow.keras.applications import VGG16

base = VGG16(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
base.trainable = False    # congelar las capas convolucionales

modelo_latas = Sequential([
    # Data augmentation (solo se activa en training)
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomBrightness(0.1),
    # Backbone preentrenado
    base,
    # Cabezal nuevo (que SI entrenamos)
    layers.Flatten(),
    layers.Dropout(0.4),                                # regularizacion
    layers.Dense(32, activation="relu"),
    layers.Dense(len(class_names), activation="softmax"),
])

modelo_latas.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",       # etiquetas enteros 0,1
    metrics=["accuracy"],
)
modelo_latas.summary()

In [ ]:
# Entrenar (con tan poca data va rápido incluso en CPU)
history = modelo_latas.fit(
    X_tr, y_tr,
    epochs=12,
    batch_size=8,
    validation_split=0.15,
    verbose=2,
)

acc_te = modelo_latas.evaluate(X_te, y_te, verbose=0)[1]
print(f"\nAccuracy en test: {acc_te:.3f}")

In [ ]:
# Curva de loss y accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train", color="#C82B40")
axes[0].plot(history.history["val_loss"], label="val", color="#2563EB")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss")
axes[1].plot(history.history["accuracy"], label="train", color="#C82B40")
axes[1].plot(history.history["val_accuracy"], label="val", color="#2563EB")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
axes[1].set_title("Accuracy")
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix sobre test
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred = modelo_latas.predict(X_te, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title(f"Confusion Matrix (acc={acc_te:.0%})", fontweight="bold")
plt.tight_layout(); plt.show()
print(classification_report(y_te, y_pred, target_names=class_names))

### 9.5 App de predicción

Ahora la segunda app: recibe una foto (webcam o subida), la pasa por el modelo y devuelve las probabilidades.

In [ ]:
def predecir(imagen):
    if imagen is None:
        return None
    # Mismo preprocesamiento que en entrenamiento: resize + normalizar
    img_resized = tf.image.resize(imagen, (96, 96)).numpy() / 255.0
    img_batch = img_resized[None, ...]                   # agregar dim batch
    probs = modelo_latas.predict(img_batch, verbose=0)[0]
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

with gr.Blocks(title="Clasificador de latas") as predictor:
    gr.Markdown("# Clasificador: Coca-Cola vs Pepsi")
    gr.Markdown("Toma una foto de una lata o sube una imagen. El modelo predice la marca.")
    with gr.Row():
        cam = gr.Image(sources=["webcam", "upload"], type="numpy", label="Foto")
        out = gr.Label(num_top_classes=2, label="Predicción")
    cam.change(predecir, inputs=cam, outputs=out)

predictor.launch(share=True, debug=False)

### 9.6 ¿Qué hacer si el modelo no funciona bien?

**Si val_accuracy < 70%:**
- **Pocas fotos** → recolectar 50+ por clase
- **Sesgo de fondo** → variar más los fondos
- **Dataset desbalanceado** → balancear con `class_weight` o más fotos de la clase pobre

**Si val_accuracy alto pero falla en fotos nuevas:**
- **Overfitting** → subir Dropout, agregar más augmentation, recolectar más fotos
- **Distribución diferente** → tus fotos de test fueron tomadas en otro contexto que las de train

**Si confunde Coca con Pepsi específicamente:**
- Tomar fotos enfocadas en la **marca** (logo, color rojo vs azul)
- Mirar la confusion matrix: ¿en qué dirección falla más?

### Entregable de la actividad

1. Link público de Gradio (sale del `share=True`)
2. Screenshot de la confusion matrix y curvas de loss
3. Explicación: cuántas fotos por clase, qué accuracy lograste, qué errores observaste

---
## Resumen

| Concepto | Detalle |
|---|---|
| **Problema del MLP** | No invariante a traslación, explota en parámetros, ignora estructura espacial |
| **Idea CNN** | Filtros 3×3 con pesos compartidos que recorren la imagen |
| **Conv2D** | Aplica N filtros aprendidos → N mapas de características |
| **MaxPooling** | Reduce a la mitad cada dimensión (sin parámetros) |
| **Arquitectura típica** | 2-3 bloques (Conv + Pool), Flatten, 1-2 Dense |
| **Keras** | API alto nivel: `model.fit()` igual que sklearn |
| **Backend** | TensorFlow / JAX / PyTorch hace los gradientes |
| **Data augmentation** | Capa que genera variaciones en training (más datos gratis) |
| **Transfer learning** | Reusar CNN preentrenada → solo entrenar cabezal |
| **Cuándo no CNN** | Tabular, texto, series temporales no espaciales |
| **Hoy en visión** | ViT compite, pero CNN sigue siendo la opción práctica |

**La idea más importante:** las redes aprenden **representaciones jerárquicas** (bordes → texturas → partes → objetos) directamente de los datos. Esa idea mueve toda la IA moderna.

**Siguiente módulo:** IA Generativa (autoencoders, VAE, GANs, diffusion). Estos mismos bloques se usan para *generar* en vez de *clasificar*.